In [1]:
import numpy as np
import cloudvolume
from tqdm import tqdm
import glob
import os
import json

In [2]:
# EM 
em_seg_batch_paths = [
    [   # mrgd dataset .. ignore
        "precomputed://gs://htem_dorsalhorn_mrgd_r1093/dorsal_sections/dorsal_sections_0-500",
        "graphene://https://cave.fanc-fly.com/segmentation/table/mouse_dorsal_spine"
    ],
    [   # spinal-cord dataset
        "precomputed://gs://lee-mouse-spinal-cord-001-raw/sharded",
        "graphene://https://cave.fanc-fly.com/segmentation/table/wclee_mouse_spinalcord_cltmr"
    ]
]

DATASET_IDX = 1
MIP = (32,32,45)

em_path = em_seg_batch_paths[DATASET_IDX][0]
seg_path = em_seg_batch_paths[DATASET_IDX][1]

spinalcord_em = cloudvolume.CloudVolume(em_path, use_https=True, 
                                  fill_missing=True,
                                  mip=MIP,
                                  progress=False)
# SEG
spinalcord_seg = cloudvolume.CloudVolume(seg_path, 
                                   use_https=True, #secrets='REPLACE_WITH_YOUR_CAVE_TOKEN', 
                                   mip=MIP, 
                                   agglomerate=True, # for rootids
                                   progress=False)

In [3]:
def get_seg_id_from_coord(coord, seg_mip=(32,32,45), coordinate_mip=(32,32,45)):

    mip_scale = np.array(seg_mip)/np.array(coordinate_mip)
    seg_coord = (coord[0]//mip_scale[0], coord[1]//mip_scale[1], coord[2]//mip_scale[2])
    
    voxel_seg_id = spinalcord_seg[seg_coord] # segmentation ID of neuron coordinate
    return voxel_seg_id

In [4]:
import pandas as pd
from pathlib import Path
import re

# === Setup ===
folder_path = Path('/Users/wangchu/v3/v4/v4_gnn/v4_gnn')

# Define all cell types you want columns for (include low_confidence_sensory)
cell_types = ['Aδ-HTMR', 'C-Cold', 'C-HTMR', 'C-HTMR-Non-peptidergic',
              'C-Heat', 'N/A', 'Aβ-LTMR', 'Aδ-LTMR', 'C-LTMR', 'low_confidence_sensory']

# Classifier columns used to assess low-confidence sensory identity
classifier_column = ['cltmr', 'abltmr', 'adltmr', 'chtmrnp',
                   'c-htmr', 'adhtmrp', 'cheat', 'cold']

# Regex pattern to extract coordinates from filename
pattern = re.compile(r'cell_type_predictions_(\d+)_(\d+)_(\d+)\.feather')

# === Build rows ===
rows = []

for feather_file in folder_path.glob('cell_type_predictions_*.feather'):
    match = pattern.search(feather_file.name)
    if not match:
        continue

    x, y, z = map(int, match.groups())
    vol = get_seg_id_from_coord((x, y, z), seg_mip=(32, 32, 45), coordinate_mip=(32, 32, 45))
    root_id = int(vol[0, 0, 0, 0])

    df = pd.read_feather(feather_file)

    # Create mask for is_sensory == True
    is_sensory_mask = df['is_sensory'] == True

    # Create mask for all classifier values <= 0.9
    low_confidence_mask = (df[classifier_column] <= 0.6).all(axis=1)

    # Combine masks
    final_mask = is_sensory_mask & low_confidence_mask

    # Update 'cell_type' for those rows
    df.loc[final_mask, 'cell_type'] = 'low_confidence_sensory'
    # Step 2: Count cell_type occurrences
    synapse_counts = df['cell_type'].value_counts().to_dict()

    # Step 3: Build row for result_df
    row_data = {ct: synapse_counts.get(ct, 0) for ct in cell_types}
    row_data['neuron_id'] = root_id
    rows.append(row_data)

# === Create result_df ===
result_df = pd.DataFrame(rows).set_index('neuron_id')
print(result_df.head())


                    Aδ-HTMR  C-Cold  C-HTMR  C-HTMR-Non-peptidergic  C-Heat  \
neuron_id                                                                     
720575940905042134        8       7       1                      10      13   
720575940892190818        1       0       2                       6       3   
720575940882124516        2       0       0                      33       0   
720575940893395733       14       0       5                     778       8   
720575940940386475       15      34       3                       1      12   

                     N/A  Aβ-LTMR  Aδ-LTMR  C-LTMR  low_confidence_sensory  
neuron_id                                                                   
720575940905042134  2967        2        0       2                      14  
720575940892190818   991       70      419      46                      12  
720575940882124516   620        1        2     704                       7  
720575940893395733  1168        0        0      15           

In [5]:
# Libraries
import caveclient
import subprocess
import json
import numpy as np
import matplotlib.pyplot as plt
# Setup caveclient
client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr", ) #server_address='https://global.daf-apis.com')
# Synapses df
cell_type = client.materialize.query_table('spinalcord_neuron_clusters')

Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


In [6]:
# Ensure IDs are strings
cell_type['pt_root_id'] = cell_type['pt_root_id'].astype(str)
result_df.index = result_df.index.astype(str)

# Build base mapping from pt_root_id to tag
id_to_tag = dict(zip(cell_type['pt_root_id'], cell_type['tag']))

# Map each root_id in result_df to its tag
result_df['cell_type'] = result_df.index.map(id_to_tag)


In [8]:
set(result_df['cell_type'])

{'adltmr_module',
 'cltmr_module',
 nan,
 'noci_module',
 'np_module',
 'p1',
 'p2_cold',
 'p2_noci',
 'p2_poly',
 'r',
 'v1',
 'v2'}

In [9]:
# Step 1: Define which cell types to include
excitatory_types = ['p1','p2_cold','p2_noci','p2_poly']

# Step 2: Define the input columns to use for filtering
input_columns = ['C-Cold', 'Aδ-HTMR', 'C-Heat', 'C-HTMR',
                 'C-HTMR-Non-peptidergic', 'C-LTMR',
                 'Aδ-LTMR', 'Aβ-LTMR']

# Step 3: Subset from result_df
excitatory_neurons = result_df[result_df['cell_type'].isin(excitatory_types)].copy()

# Step 4: Compute row-wise totals and thresholds (10%)
row_totals = excitatory_neurons[input_columns].sum(axis=1)
thresholds = row_totals * 0

# Step 5: Apply threshold filtering
excitatory_neurons[input_columns] = excitatory_neurons[input_columns].where(
    excitatory_neurons[input_columns].ge(thresholds, axis=0),
    0
)

In [10]:
excitatory_neurons['Aδ-HTMR + C-Heat'] = (
    excitatory_neurons['Aδ-HTMR'] + excitatory_neurons['C-Heat']
).fillna(0)

In [11]:
excitatory_neurons

,Aδ-HTMR,C-Cold,C-HTMR,C-HTMR-Non-peptidergic,C-Heat,N/A,Aβ-LTMR,Aδ-LTMR,C-LTMR,low_confidence_sensory,cell_type,Aδ-HTMR + C-Heat
neuron_id,,,,,,,,,,,,
720575940940386475,15,34,3,1,12,4819,8,6,15,24,p1,27
720575940914014910,59,93,3,15,82,2232,0,0,6,20,p2_poly,141
720575940889952699,32,111,4,0,98,1668,0,0,0,15,p2_poly,130
720575940888802138,65,129,3,1,23,5276,2,0,2,17,p1,88
720575940882393805,13,31,2,0,4,3101,0,0,0,5,p1,17
720575940902518994,78,40,70,73,186,2920,0,0,13,48,p2_poly,264
720575940898432765,390,16,32,6,255,2590,0,0,0,41,p2_noci,645
720575940894563284,426,98,1,0,230,1304,0,0,0,61,p2_noci,656
720575940927796016,50,545,1,0,101,1138,0,0,0,42,p2_cold,151


In [13]:
# Export your dataframe (replace `df` with your actual dataframe variable name)
out_path = "/Users/wangchu/mesh_analysis/sensory_inputs/projection_neuron.csv"
excitatory_neurons.to_csv(out_path, index=True)
print(f"Saved CSV to: {out_path}")


Saved CSV to: /Users/wangchu/mesh_analysis/sensory_inputs/projection_neuron.csv
